In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os
import shutil
import random

In [5]:
def split_dataset(src, train_dst, val_dst, split_ratio=0.8):
    classes = ['cat', 'dog']
    for cls in classes:
        img_dir = os.path.join(src, cls)
        images = os.listdir(img_dir)
        random.shuffle(images)

        split = int(len(images) * split_ratio)
        train_imgs = images[:split]
        val_imgs = images[split:]

        os.makedirs(os.path.join(train_dst, cls), exist_ok=True)
        os.makedirs(os.path.join(val_dst, cls), exist_ok=True)
        
        for img in train_imgs:
            shutil.copy(os.path.join(img_dir, img), os.path.join(train_dst, cls, img))
        for img in val_imgs:
            shutil.copy(os.path.join(img_dir, img), os.path.join(val_dst, cls, img))

split_dataset(
    src='kagglecatsanddogs/all_images', 
    train_dst='kagglecatsanddogs/train',
    val_dst='kagglecatsanddogs/val'
)

In [6]:
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder('kagglecatsanddogs/train', transform=transform_train)
val_dataset = datasets.ImageFolder('kagglecatsanddogs/val', transform=transform_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [8]:
resnet50 = models.resnet50(pretrained=True)

for param in resnet50.parameters():
    param.requires_grad = False

resnet50.fc = nn.Linear(2048, 1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet50 = resnet50.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(resnet50.fc.parameters(), lr=0.001)

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /home/helia/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [01:28<00:00, 1.16MB/s]


In [9]:
for epoch in range(5):
    resnet50.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = resnet50(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {running_loss / len(train_loader):.4f}")

/home/helia/Desktop/code/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch 1, Loss: 0.1266
Epoch 2, Loss: 0.0789
Epoch 3, Loss: 0.0743
Epoch 4, Loss: 0.0699
Epoch 5, Loss: 0.0677


In [10]:
resnet50.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = resnet50(images)
        preds = (outputs > 0.5).squeeze().long()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

torch.save(resnet50.state_dict(), "resnet50_dogs_vs_cats.pth")

print(f"Validation Accuracy: {100 * correct / total:.2f}%")

Validation Accuracy: 98.34%


In [12]:
from PIL import Image
import torchvision.transforms as transforms

In [17]:
def predict_image(image_path, model, device):
    img = Image.open(image_path).convert('RGB')
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])
    input_tensor = transform(img).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        output = model(input_tensor)
        prob = torch.sigmoid(output).item()
    label = "Dog" 
    if prob < 0.5 :
        label = "Cat"
        prob = 1 - prob
    print(f"Prediction: {label} ({prob:.2f})")


In [ ]:
predict_image('kagglecatsanddogs/testing/sample1.jpg', resnet50, device)
predict_image('kagglecatsanddogs/testing/sample2.jpg', resnet50, device)
predict_image('kagglecatsanddogs/testing/sample3.jpg', resnet50, device)
predict_image('kagglecatsanddogs/testing/sample5.jpg', resnet50, device)
predict_image('kagglecatsanddogs/testing/sample6.jpg', resnet50, device)
predict_image('kagglecatsanddogs/testing/sample7.jpg', resnet50, device)
predict_image('kagglecatsanddogs/testing/sample8.jpg', resnet50, device)


Prediction: Cat (0.73)
Prediction: Dog (0.93)
Prediction: Dog (0.83)
Prediction: Cat (0.88)
Prediction: Cat (1.00)
Prediction: Dog (1.00)
Prediction: Cat (0.99)
Prediction: Dog (0.75)
Prediction: Cat (0.69)
Prediction: Dog (0.85)
Prediction: Dog (0.91)
